# Synthetic biomarker benchmark — the shapes and what they settle

This notebook is the specification. Every shape below has values that follow from its geometry, and
those values are what a biomarker implementation is measured against — not another implementation's
opinion, and not a human's tracing.

**Why shapes at all.** A biomarker's name is not its definition: "tortuosity" names at least three
incompatible formulas, and papers usually report the name and omit the choice. Comparing two
programs tells you they differ without telling you which is right. A straight vessel has tortuosity
exactly 1, and that settles it.

**What it catches that a dataset cannot:** a factor of two in a curvature formula, a centreline
extractor that counts pixel steps where it should measure arc length, an arteriovenous ratio
computed from arteries alone, a measurement that moves when the vessel is rotated.

Rotations are generated by the run, not shown here — this notebook is the shapes at rest.

How the benchmark is configured: [biomarker-synthetic-docs.md](../docs/benchmarks/biomarker-synthetic-docs.md).
The rules the shapes follow: `.claude/skills/build-benchmark/biomarker-synthetic.md`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def repository() -> Path:
    """The repository, found rather than assumed."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))

from benchmarks.shapes import library  # noqa: E402

#: The grid these pictures are drawn at. The run uses several; the shapes are the same at each.
SIDE = 1024

built = {name: library.build(name, side=SIDE) for name in library.SHAPES}
print(f"{len(built)} shapes at {SIDE}²: {', '.join(built)}")

## 1. The shapes

Each is drawn from a centreline in continuous coordinates and rasterised once — never drawn small
and enlarged, and never rotated as a picture, because resampling a structure a few pixels wide
destroys it.

**Arteries are red, veins are blue**, the field of view is the dark circle, and the optic disc is
the ring. A shape that tests a measurement over the vessels as one class puts them all in the
artery mask and leaves the vein mask absent — an adapter then forms the union of what it was
given, which is the same derivation the artery/vein benchmark applies to every model.

In [ ]:
def picture(shape, axis) -> None:
    """One shape as an adapter receives it: two classes, a field of view, and where the disc is."""
    canvas = np.zeros((shape.side, shape.side, 3))
    canvas[shape.fov] = 0.12
    if shape.artery is not None:
        canvas[shape.artery] = (0.85, 0.15, 0.15)
    if shape.vein is not None:
        canvas[shape.vein] = (0.20, 0.35, 0.95)
    axis.imshow(canvas)
    x, y, radius = shape.disc
    axis.add_patch(plt.Circle((x, y), radius, fill=False, color="white", linewidth=1.2))
    classes = " + ".join(
        name for name, mask in (("artery", shape.artery), ("vein", shape.vein)) if mask is not None
    )
    axis.set_title(f"{shape.name}\n{classes}", fontsize=10)
    axis.set_xticks([])
    axis.set_yticks([])


fig, axes = plt.subplots(2, 3, figsize=(12, 8.4))
for axis, shape in zip(axes.ravel(), built.values()):
    picture(shape, axis)
fig.suptitle("Every shape, at rest", fontsize=12)
fig.tight_layout()

## 2. What each shape settles

One row per quantity, one column per shape, keyed as `biomarker/variant` after the catalogue in
`docs/biomarkers/` — so a number an implementation returns is compared with the right definition
rather than with whichever happens to share its name.

An empty cell means the shape says nothing about that quantity, which is not the same as zero.

In [ ]:
theory = pd.DataFrame({name: shape.theory for name, shape in built.items()})
theory.index.name = "biomarker / variant"
theory.round(4)

## 3. The derivations

Each value above follows from the geometry, and each is stated here so that a reader can disagree
with the arithmetic rather than with the code.

### 3.1 Straight vessel

Tortuosity is **exactly 1** under any arc-over-chord definition, because the arc *is* the chord.
Total curvature is **exactly 0**. The calibre is the width it was drawn at. Its area is the width
times the length, plus a disc of that width for the two rounded ends — the two-dimensional tube
formula, which holds for any curve that does not cross itself.

This is the shape that catches the most basic error available: an implementation whose tortuosity
is not 1 here is not computing tortuosity.

### 3.2 Circular arc

Curvature is **exactly 1/r** everywhere. Arc length is `rθ`, the chord is `2r sin(θ/2)`, so

$$\tau_1 = \frac{r\theta}{2r\sin(\theta/2)} = \frac{\theta}{2\sin(\theta/2)}$$

in which **`r` cancels**. That is the useful part: τ1 depends on how far the vessel bends and not at
all on how large it is, so an implementation whose τ1 changes with the radius is computing
something else. Total curvature is `θ`; total squared curvature is `θ/r`.

### 3.3 Sinusoid

Its arc length is an elliptic integral with no elementary closed form, so the theory *is* the
integral — evaluated over 200,001 points, which is exact to far more places than any implementation
will reach. The same integration gives `∫κ ds` and `∫κ² ds`, with

$$\kappa = \frac{|y''|}{(1 + y'^2)^{3/2}}$$

A curve with curvature that varies along its length is what separates an implementation that
integrates properly from one that averages a few sampled points.

### 3.4 Bifurcation

Symmetric, so the angle between the daughters is exactly the angle asked for and neither daughter
is the trunk. One junction, three endpoints, one connected component.

### 3.5 Parallel vessels that never meet

No junctions, and as many components as there are lines. A skeletoniser that joins them, or a
junction counter that finds a crossing where two vessels merely pass near one another, says so here
and nowhere else.

### 3.6 Artery beside vein

The only shape that draws both classes, and the only one that can test a ratio of the two. Widths
`wa` and `wv` are constant, so the arteriovenous ratio is **exactly `wa / wv`** under any variant
that is a ratio of calibres.

It also separates two things the catalogue records as different variants without anyone having
measured the difference: an implementation that divides two calibres, and one that computes two
central retinal equivalents over a ring around the disc and divides those. On parallel vessels of
constant width those are the same number; on a real eye they are not.

## 4. A rasterised shape is not the shape

The theory describes a continuous curve; an implementation sees pixels. The difference is real and
is nobody's bug — and it is why this benchmark renders every shape at several resolutions rather
than judging at one.

Below: the area actually drawn, against the area the geometry requires, as the grid refines.

**Expect a sawtooth, not a smooth curve.** What separates the two is at most one row of pixels along
the vessel, and whether that row is taken depends on where the centreline falls between pixel
centres. The error can therefore rise between two resolutions with nothing wrong. Read the trend
across four, never the step between two.

In [ ]:
rows = []
for side in (256, 512, 1024, 2048):
    shape = library.build("straight", side=side)
    drawn = float(shape.artery.sum())
    rows.append(
        {
            "side": side,
            "width, px": round(shape.parameters["width"], 1),
            "area drawn": drawn,
            "area geometry requires": shape.theory["vessel-area-and-length/area"],
            "relative difference": drawn / shape.theory["vessel-area-and-length/area"] - 1.0,
            "one row of pixels would be": 1.0 / shape.parameters["width"],
        }
    )
pd.DataFrame(rows).set_index("side").round(4)

## 5. What this benchmark does not test

- **Cup-to-disc ratio.** An adapter is handed the disc as a centre and a radius and is given no
  cup, so the ratio cannot be formed. It is measured against ophthalmologists' own outlines in the
  [disc benchmark](../docs/benchmarks/disc-results.md) instead.
- **Anything anchored to the fovea** — the disc-fovea distance, and the temporal angle — because a
  synthetic shape has no macula and inventing one would make the answer a property of the fixture.
- **Vessel tracing**, which is a step rather than a number.
- **Whether an implementation works on a photograph.** A program that measures a synthetic vessel
  exactly may still fail on a segmentation of a real eye, which is the next benchmark's question.